In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.



# Create a Headless IsaacSim

You should only have to do this part once; it will load kit, or at least the headless IsaacSim version of kit, by calling
simulation_app = SimulationApp(CONFIG).This will take some time, so give it a minute and wait for it to say "Hi".


In [ ]:
from isaacsim import SimulationApp

simulation_app = SimulationApp()
print("Hi")

# Any Omniverse level imports must occur after the `SimulationApp` class is instantiated (because APIs are provided
# by the extension/runtime plugin system, it must be loaded before they will be available to import).
import isaacsim.core.experimental.utils.app as app_utils
import isaacsim.core.experimental.utils.semantics as semantics_utils
import isaacsim.core.experimental.utils.stage as stage_utils
from isaacsim.asset.importer.urdf import URDFImporter, URDFImporterConfig
from isaacsim.core.experimental.materials import OmniGlassMaterial
from isaacsim.core.experimental.prims import XformPrim
from isaacsim.core.rendering_manager import ViewportManager
from isaacsim.storage.native import get_assets_root_path

import omni
import carb
import numpy as np


## Creating the scene

Re-run the cell below to randomize the scene from scratch. The goal here is to make iterating on scene setup easy and not require restarts of the omniverse application.


In [ ]:
# SCENE SETUP
stage_utils.create_new_stage(template="gridroom")
ViewportManager.set_camera_view(
    ViewportManager.get_camera(),
    eye=[-0.9025, 2.1035, 1.0222],
    target=[0.6039, 0.30, 0.0950],
)

# Load a URDF: convert it to USD and add it as a reference to the current stage
extension_path = app_utils.get_extension_path("isaacsim.asset.importer.urdf")
importer = URDFImporter(
    config=URDFImporterConfig(
        urdf_path=extension_path + "/data/urdf/robots/carter/urdf/carter.urdf",
        merge_fixed_joints=False,
        fix_base=False,
    )
)
robot_usd_path = importer.import_urdf()

robot_prim_path = "/Robot"
stage_utils.add_reference_to_stage(usd_path=robot_usd_path, path=robot_prim_path)
semantics_utils.add_labels(robot_prim_path, labels=["Robot"], taxonomy="class")

# Load a mesh
assets_root_path = get_assets_root_path()
if assets_root_path is None:
    carb.log_error("Could not find Isaac Sim assets folder")
usd_path = assets_root_path + "/Isaac/Props/YCB/Axis_Aligned/006_mustard_bottle.usd"

stage_utils.add_reference_to_stage(usd_path=usd_path, path="/Mesh")
semantics_utils.add_labels("/Mesh", labels=["mustard"], taxonomy="class")
xform_prim = XformPrim(paths="/Mesh", positions=[1.0, 0, 0], scales=[10.0, 10.0, 10.0], reset_xform_op_properties=True)

# Apply a glass material to mesh
material = OmniGlassMaterial("/Looks/GlassMaterial")
material.set_input_values("glass_ior", 1.25)
material.set_input_values("depth", 0.001)
material.set_input_values("thin_walled", False)
material.set_input_values("glass_color", np.random.rand(3).tolist())
xform_prim.apply_visual_materials(material)


## Viewing the scene in the notebook

This next example does not change the scene (but it could if you used commands like the ones above), but it does visualize the synthetic data by accesing the underlying annotator data. Specifically, it shows a color, depth, and segmentation view of the scene, and then displays them within the notebook.


In [ ]:
import matplotlib.pyplot as plt
from omni.syntheticdata import visualize
from omni.kit.viewport.utility import get_active_viewport
import omni.replicator.core as rep

viewport_api = get_active_viewport()
active_cam = viewport_api.get_active_camera()
resolution = viewport_api.get_texture_resolution()
render_product = rep.create.render_product(active_cam, resolution)

rgb = rep.AnnotatorRegistry.get_annotator("rgb")
rgb.attach([render_product])
depth = rep.AnnotatorRegistry.get_annotator("distance_to_image_plane")
depth.attach([render_product])
semantic_segmentation = rep.AnnotatorRegistry.get_annotator("semantic_segmentation")
semantic_segmentation.attach([render_product])

# Run the application for multiple frames to ensure the synthetic data pipeline is initialized
timeline = omni.timeline.get_timeline_interface()
timeline.play()
for _ in range(10):
    simulation_app.update()
timeline.pause()

# Get groundtruth
rgb_data = rgb.get_data()
depth_data = depth.get_data()
semantic_segmentation_data = semantic_segmentation.get_data()

# GROUNDTRUTH VISUALIZATION
# Setup a figure
_, axes = plt.subplots(1, 3, figsize=(20, 7))
axes = axes.flat
for ax in axes:
    ax.axis("off")

# RGB
axes[0].set_title("RGB")
axes[0].imshow(rgb_data)

# DEPTH
axes[1].set_title("Depth")
depth_data_clipped = np.clip(depth_data, 0, 255)
axes[1].imshow(visualize.colorize_distance(depth_data.squeeze()))

# SEMANTIC SEGMENTATION
axes[2].set_title("Semantic Segmentation")
# Draw the segmentation mask on top of the color image with a transparency
axes[2].imshow(rgb_data)
semantic_segmentation_rgb = visualize.colorize_segmentation(semantic_segmentation_data["data"])
axes[2].imshow(semantic_segmentation_rgb, alpha=0.7)

In [ ]:
# Cleanup application
simulation_app.close()
